# Complete sessions → discretized (x, y, z, t)

Loads preprocessed ADS-B session data, keeps **geometrically complete** sessions, bins lon/lat/alt and session-relative time, and writes `complete_sessions.json` for [`session_space_viz.html`](session_space_viz.html).

complete** = labeled `aircraft_session_id` with a valid start and end event:

| kind | start | end |
|---|---|---|
| local | in-sector/airport takeoff | in-sector/airport landing |
| departure | in-sector/airport takeoff | edge or collection end |
| arrival | edge or collection start | in-sector/airport landing |
| transit | edge or collection start | edge or collection end, no return |

Airport takeoff/landing events are accepted as complete endpoints when they have a matching airport identifier; boundary crossings use the preprocessor edge band. Dropped: unlabeled rows, parked/ground-only, too few points, collection-window clips, and unresolved interior endpoints.

The configured sector and edge widths are taken directly from `01-label_aircraft_id.ipynb`: 35–45°N, 78–68°W, with 0.4° latitude and 0.5° longitude edge bands. Observed data extrema are never assumed to be sector boundaries. If no observations reach the configured edge bands, the resulting arrival/departure/transit classifications should be considered unavailable rather than inferred from the dataset envelope.

The loader searches for a readable Feather file and skips corrupt/truncated candidates. If needed, set `FEATHER_PATH` below to an explicit preprocessed file.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

HERE = Path.cwd()
if (HERE / "complete_sessions.py").exists():
    sys.path.insert(0, str(HERE))
else:
    sys.path.insert(0, str(HERE / "03-Visualization"))

from complete_sessions import (
    DEFAULT_BINS,
    completeness_counts,
    complete_session_ids,
    discretize_sessions,
    export_complete_sessions,
    load_adsb,
)

FEATHER_PATH = None  # optionally set to Path(r'C:/.../full_ac_data.feather')

In [ ]:
df, source = load_adsb(FEATHER_PATH)
print("feather:", source)
print("rows:", f"{len(df):,}")
print("labeled sessions:", df["aircraft_session_id"].nunique(dropna=True))
print(df["aircraft_session_type"].value_counts(dropna=False))

In [ ]:
keep, summary, box = complete_session_ids(df)
counts, drops = completeness_counts(summary)
print("configured sector and edge bands:")
display(pd.Series(box))
print("complete kinds:")
display(counts)
print("dropped:")
display(drops)
print("kept:", f"{len(keep):,}")

In [ ]:
# Show several session paths for one aircraft, including original session labels.
# This reveals whether IDs are split across continuous tracks or merged across long gaps.
aircraft_counts = df.groupby("aircraft")["aircraft_session_id"].nunique(dropna=True)
sample_aircraft = aircraft_counts[aircraft_counts >= 2].sort_values(ascending=False).index[0]
ac = df.loc[df["aircraft"] == sample_aircraft].sort_values("ts")
fig, ax = plt.subplots(figsize=(10, 7))
for sid, tr in ac.groupby("aircraft_session_id", sort=False, dropna=False):
    label = str(sid) if pd.notna(sid) else "unlabeled"
    ax.plot(tr["lon"], tr["lat"], marker=".", ms=2, lw=1, label=label)
    mid = tr.iloc[len(tr) // 2]
    ax.annotate(label, (mid["lon"], mid["lat"]), fontsize=7)
ax.scatter(ac.groupby("aircraft_session_id", dropna=False).first()["lon"], ac.groupby("aircraft_session_id", dropna=False).first()["lat"], marker="o", s=22, facecolors="none", edgecolors="white", label="session start")
ax.set_xlabel("lon (x)")
ax.set_ylabel("lat (y)")
ax.set_aspect("equal", adjustable="box")
ax.legend(fontsize=7, ncol=2)
ax.set_title(f"Aircraft {sample_aircraft}: trajectories by aircraft_session_id")
plt.show()

# Compare preprocessor id (gap/event based) with aircraft_session_id.
display(ac.groupby("aircraft_session_id", dropna=False).agg(
    rows=("ts", "size"), id_count=("id", "nunique"),
    t_start=("ts", "min"), t_end=("ts", "max"),
    takeoffs=("takeoff", "sum"), touchdowns=("touchdown", "sum"),
).reset_index())

# Inspect most frequent session assignments and collection-level event counters.
display(summary.groupby(["session_type", "complete_kind", "drop_reason"], dropna=False).size().rename("sessions").reset_index())
print("Rows in configured edge bands:", int((df["lon"].lt(box["lon_min"] + box["lon_range"]) | df["lon"].gt(box["lon_max"] - box["lon_range"]) | df["lat"].lt(box["lat_min"] + box["lat_range"]) | df["lat"].gt(box["lat_max"] - box["lat_range"])).sum()))

In [ ]:
BINS = dict(DEFAULT_BINS)  # dx deg, dy deg, dz ft, dt s
occ = discretize_sessions(df, keep, bins=BINS, box=box)
print(occ.head())
print("occupancy rows:", f"{len(occ):,}")
print("t_rel seconds max:", float(occ["t"].max()))

In [ ]:
out, summary_out, meta = export_complete_sessions(
    feather_path=FEATHER_PATH,
    out_path=Path("complete_sessions.json") if (HERE / "complete_sessions.py").exists()
    else HERE / "03-Visualization" / "complete_sessions.json",
    bins=BINS,
)
print(meta)
print("wrote", out.resolve())

Open the player (fetch of the JSON fails on `file://`; serve the folder or use **Load JSON**):

```bash
cd 03-Visualization
python -m http.server 8765
```

Then open http://127.0.0.1:8765/session_space_viz.html

- 2D: occupied (x, y) at the current discretized t, fill color = z. White outline + fan dots distinguish multiple aircraft sharing (x, y, t); click an aircraft row to re-zero at its exact (x, y, z) voxel.
- 3D: drag to orbit, scroll to zoom, click a point to re-zero. All tracks through that voxel get t' = t − t_voxel and (x', y', z') relative to the voxel, with t' = 0 at the first voxel visit.